Loads Network performance data from ES. Learns to predict bandwidth from packet loss and one way delay.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import re,time
from elasticsearch import Elasticsearch, exceptions as es_exceptions
from elasticsearch.helpers import scan
es = Elasticsearch(hosts=[{'host':'atlas-kibana.mwt2.org', 'port':9200}],timeout=60)


In [ ]:
ind='network_weather_2-2016.6.22'

currentTime=int(round(datetime.utcnow().timestamp()* 1000))
lastHours=24
startTime=currentTime - lastHours*3600000 
endTime=currentTime

source='MWT2'
destination='AGLT2'

In [ ]:
myquery={
#    "size":50,
    "fields": ["timestamp", "delay_mean","throughput","packet_loss"],
    "query":{
        "filtered":{
            "query":{
                "bool" : {
                  "must" :     [ 
                        {"term": {"srcProduction": {"value": True } } },
                        {"term": {"destProduction": {"value": True } } },
                        {"term": {"srcSite": { "value": source } } },
                        {"term": {"destSite": { "value": destination } } }
                    ]
                }
#               "query_string":{"query":queryString,"analyze_wildcard":True,"lowercase_expanded_terms":False}
            }
#            ,"filter":{ "range":{"timestamp":{"gte": startTime,"lte": endTime,"format": "epoch_millis"}}   }
        }
    }
#    ,"sort": { "timestamp": { "order": "asc" }}
}

In [ ]:
scroll = scan(es, query=myquery, index=ind, scroll='10s')
RawData={}
count=0
for res in scroll:
    #print(res)
    r = res['fields']
    sts = r['timestamp'][0]
    ts=time.mktime(datetime.strptime(sts, "%Y-%m-%dT%H:%M:%S").timetuple())
    if not ts in RawData:
        RawData[ts]=[-1,-1,-1]
    if res['_type']=='latency':
        RawData[ts][0]=r['delay_mean'][0]
    elif res['_type']=='packet_loss_rate':
        RawData[ts][1]=r['packet_loss'][0]
    elif res['_type']=='throughput':
        RawData[ts][2]=r['throughput'][0]
    count+=1
    
print ("results fetched:", count ) 
print ("results summed up:", len(RawData) ) 

tss = sorted(RawData.keys())
for r in tss:
    print(r, RawData[r])


cleaning the data - create array where all 3 components are there.

In [ ]:
X=[]
y=[]
for (k,v) in RawData.items():
    if v[2]==-1: continue
    delay_measurements=0
    pl_measurements=0
    delay_sum=0
    pl_sum=0
    for (k1,v1) in RawData.items():
        if abs(k1-k)<600:
            if v1[0]>-1:
                delay_measurements+=1
                delay_sum+=v1[0]
            if v1[1]>-1:
                pl_measurements+=1
                pl_sum+=v1[1]
            #print(k, k1)
    if pl_measurements>0 and delay_measurements>0:
#         print (k, v, delay_sum/delay_measurements, pl_sum/pl_measurements)
        X.append([ delay_sum/delay_measurements, pl_sum/pl_measurements ])
        y.append(v[2]/1024/1024/1024/8)
#print(X)
#print(y)

takes as input two arrays: an array X of size [n_samples, n_features] holding the training samples, and an array y of class labels (strings or integers), size [n_samples]

Learning step:

In [ ]:
from sklearn import svm
# X = [[0, 0], [1, 1], [1, 1.2], [0.5, 0.8]]
#y = [1.2, 2.2]
clf = svm.SVR()
f = clf.fit(X, y)
print(f)


Using what was learned

In [ ]:
clf.predict([[1., .02],[0.1,1.0]])

plot values

In [ ]:
na=np.array(X)
#print(na)

In [ ]:
owd=na[:,0]
pl=na[:,1]
ar=np.array(y)*100

In [ ]:
area = np.pi * 6**2 # 0 to 15 point radiuses
plt.figure(figsize=[18, 8])
plt.scatter(owd, pl, s= ar, c=y, alpha=0.5)
plt.colorbar()
plt.xlabel('One way delay [ms]'), plt.ylabel('packet loss')
plt.show()